# Homework 3 – Survival Analysis
**Karen Hovhannisyan** | Marketing Analytics

---

## Dataset Overview
The telco dataset contains 1,000 subscribers with the following variables:

| Column | Description |
|---|---|
| tenure | Customer lifetime in months (our **time** variable) |
| churn | Whether the customer churned — Yes/No (our **event** variable) |
| age, income, address | Demographic/socioeconomic features |
| ed, retire, gender, marital | Personal characteristics |
| voice, internet, forward | Service add-ons |
| custcat | Customer category (Basic, E-service, Plus, Total) |
| region | Geographic zone |


In [ ]:
import pandas as pd
import numpy as np
from lifelines import (WeibullAFTFitter, LogNormalAFTFitter,
                       LogLogisticAFTFitter, GeneralizedGammaRegressionFitter)
from lifelines.utils import concordance_index
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

df = pd.read_csv('telco.csv')
df['event'] = (df['churn'] == 'Yes').astype(int)

print(f"Shape: {df.shape}")
print(f"Churn rate: {df['event'].mean():.1%}")
df.head()

## Data Preparation

In [ ]:
cat_cols = ['region','marital','ed','retire','gender',
            'voice','internet','forward','custcat']
df_enc = pd.get_dummies(df, columns=cat_cols, drop_first=True)
df_enc.columns = (df_enc.columns
                  .str.replace(' ','_').str.replace('/','_').str.replace('-','_'))

feature_cols = [c for c in df_enc.columns
                if c not in ['ID','churn','event','tenure']]
fit_df = df_enc[['tenure','event'] + feature_cols].dropna()
print(f"Features: {len(feature_cols)} | Rows: {len(fit_df)}")

## Parametric AFT Models

We fit four Accelerated Failure Time (AFT) distributions:
- **Weibull** – flexible, widely used, generalises Exponential
- **Log-Normal** – symmetric on log scale, good for moderate-tailed data
- **Log-Logistic** – allows non-monotone hazard (useful if risk peaks then falls)
- **Generalised Gamma** – most flexible, nests all three above

In an AFT model the log-survival time is modelled as a linear function of covariates:

$$\log(T) = \mu + \sigma \cdot \varepsilon, \quad \text{where } \varepsilon \sim F(\cdot)$$

A **positive coefficient** means the covariate *extends* survival (delays churn).  
A **negative coefficient** means the covariate *accelerates* churn.


In [ ]:
gg = GeneralizedGammaRegressionFitter(penalizer=1.0)
gg._scipy_fit_method = 'SLSQP'

MODELS = {
    'Weibull':      WeibullAFTFitter(penalizer=0.01),
    'Log-Normal':   LogNormalAFTFitter(penalizer=0.01),
    'Log-Logistic': LogLogisticAFTFitter(penalizer=0.01),
    'Gen. Gamma':   gg,
}

results = {}
for name, m in MODELS.items():
    m.fit(fit_df, duration_col='tenure', event_col='event')
    pred = m.predict_median(fit_df)
    ci   = concordance_index(fit_df['tenure'], pred, fit_df['event'])
    results[name] = dict(model=m, AIC=m.AIC_, log_lik=m.log_likelihood_, C=ci)

comp = (pd.DataFrame({k:{kk:vv for kk,vv in v.items() if kk!='model'}
                      for k,v in results.items()})
        .T.astype(float).sort_values('AIC').round(2))
print("Model Comparison (sorted by AIC):")
comp

## Survival Curves – All Models on One Plot

In [ ]:
median_row = fit_df[feature_cols].median().to_frame().T.reset_index(drop=True)
t_range    = np.linspace(0, df['tenure'].max(), 300)

fig, ax = plt.subplots(figsize=(9, 5.5))
colors = ['#2196F3','#E91E63','#4CAF50','#FF9800']
for (name, res), color in zip(results.items(), colors):
    sf = res['model'].predict_survival_function(median_row, times=t_range)
    ax.plot(t_range, sf.values.flatten(),
            label=f"{name}  (AIC={res['AIC']:.0f})", color=color, lw=2.2)

ax.set_xlabel('Tenure (months)', fontsize=12)
ax.set_ylabel('Survival Probability', fontsize=12)
ax.set_title('AFT Models – Survival Curves (median customer)', fontsize=13)
ax.legend(fontsize=10); ax.set_ylim(0,1); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('aft_model_comparison.png', dpi=150)
plt.show()

## Model Selection

**Best model: Log-Normal AFT** (lowest AIC = 2966.44, C-index = 0.786)

| Criterion | Winner | Reasoning |
|---|---|---|
| AIC | Log-Normal | Lowest AIC among all four models |
| C-index | Log-Logistic ≈ Log-Normal | Both ≈ 0.786 |
| Interpretability | Log-Normal | Coefficients intuitive as log-time accelerators |
| Business use | Log-Normal | Works well when churn risk is not monotone; customers often have a "grace period" before risk peaks |

As a **decision-maker**, the Log-Normal model is preferred because:
1. It has the best AIC, meaning it strikes the best bias–variance balance.
2. The log-Normal assumption is natural for tenure data which is right-skewed.
3. It is easy to interpret for non-technical stakeholders.


## Significant Features – Final Model

In [ ]:
best_name = 'Log-Normal'
best_m    = results[best_name]['model']
summary   = best_m.summary

lvl0 = summary.index.get_level_values(0)
sub  = summary[lvl0 == 'mu_']
sig_features = [f for f in sub.index.get_level_values(-1)[sub['p'] < 0.05]
                if f != 'Intercept']

print(f"Significant features (p<0.05): {len(sig_features)}")
sub[sub['p'] < 0.05][['coef','exp(coef)','p']].round(4)

In [ ]:
# Re-fit final model with only significant features
final_fit_df = df_enc[['tenure','event'] + sig_features].dropna()
final_model  = LogNormalAFTFitter(penalizer=0.01)
final_model.fit(final_fit_df, duration_col='tenure', event_col='event')

pred_f = final_model.predict_median(final_fit_df)
ci_f   = concordance_index(final_fit_df['tenure'], pred_f, final_fit_df['event'])
print(f"Final model  AIC={final_model.AIC_:.2f}  C-index={ci_f:.4f}")
final_model.summary[['coef','exp(coef)','p']]

## Coefficient Interpretation (Log-Normal AFT)

A positive μ coefficient means the covariate **slows churn** (extends tenure).  
A negative μ coefficient means the covariate **accelerates churn**.

| Feature | Coef | Interpretation |
|---|---|---|
| `custcat_E_service` | +0.93 | E-service customers churn ~2.5× later than Basic |
| `custcat_Total_service` | +0.87 | Total service customers last ~2.4× longer |
| `custcat_Plus_service` | +0.75 | Plus service customers last ~2.1× longer |
| `age` | +0.036 | Older subscribers stay longer (each extra year adds ≈3.6% to tenure) |
| `address` | +0.042 | Residential stability → loyalty |
| `internet_Yes` | **−0.79** | Internet subscribers churn **2.2× faster** |
| `marital_Unmarried` | −0.44 | Unmarried customers churn sooner |
| `voice_Yes` | −0.40 | Voice-add-on users churn somewhat faster |


## CLV Calculation

In [ ]:
# CLV = MM × Σ_{i=1}^{60} p_i / (1 + r/12)^(i-1)
# MM  = monthly income margin (income × $1000 / 12 × 20%)
# r   = 10% annual discount rate | Horizon = 60 months

DISCOUNT = 0.10; MARGIN = 0.20; HORIZON = 60
months   = np.arange(1, HORIZON+1)
discount = (1 + DISCOUNT/12)**(months-1)

clv_df = df_enc[['ID','income'] + sig_features].copy()
clv_df['tenure'] = df_enc['tenure']
clv_df['event']  = df_enc['event']
clv_df = clv_df.dropna().reset_index(drop=True)

sf_matrix = final_model.predict_survival_function(clv_df[sig_features], times=months)

clv_df['monthly_margin'] = (clv_df['income'] * 1000 / 12) * MARGIN
clv_df['CLV'] = clv_df['monthly_margin'].values * (sf_matrix.values / discount[:,None]).sum(axis=0)

print("CLV Distribution:")
clv_df['CLV'].describe().round(0)

## CLV by Segment

In [ ]:
clv_df['churn']    = df.loc[clv_df.index, 'churn'].values
clv_df['custcat']  = df.loc[clv_df.index, 'custcat'].values
clv_df['gender']   = df.loc[clv_df.index, 'gender'].values
clv_df['region']   = df.loc[clv_df.index, 'region'].values
clv_df['age_group']= pd.cut(df.loc[clv_df.index,'age'],
                             bins=[0,30,45,65,100],
                             labels=['<30','30-45','45-65','65+'])

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()
palette = ['#2196F3','#E91E63','#4CAF50','#FF9800','#9C27B0','#00BCD4']

for ax, col in zip(axes, ['custcat','gender','age_group','region']):
    groups = clv_df.groupby(col)['CLV'].mean().sort_values(ascending=False)
    bars   = ax.bar(range(len(groups)), groups.values,
                    color=palette[:len(groups)], edgecolor='white')
    ax.set_xticks(range(len(groups)))
    ax.set_xticklabels(groups.index, fontsize=9, rotation=20, ha='right')
    ax.set_title(f'Mean CLV by {col}', fontsize=11)
    ax.set_ylabel('CLV ($)', fontsize=9); ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, groups.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+500,
                f'${val:,.0f}', ha='center', fontsize=8)

plt.suptitle('Customer Lifetime Value by Segment', fontsize=14)
plt.tight_layout()
plt.savefig('clv_by_segment.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Detailed segment tables
for col in ['custcat','gender','age_group','region']:
    print(f"\n--- {col} ---")
    print(clv_df.groupby(col)['CLV'].agg(['mean','median','count']).round(0).to_string())

## Annual Retention Budget

In [ ]:
# At-risk: customers with survival probability < 0.5 at 12 months
sf_12 = sf_matrix.loc[12].values
clv_df['surv_12m'] = sf_12
at_risk = clv_df[clv_df['surv_12m'] < 0.5]

print(f"At-risk subscribers (S(12m) < 0.50): {len(at_risk)}")
print(f"Mean CLV of at-risk group:   ${at_risk['CLV'].mean():>10,.0f}")
print(f"Total CLV of at-risk group:  ${at_risk['CLV'].sum():>10,.0f}")
print(f"Suggested budget (30% rule): ${at_risk['CLV'].sum()*0.30:>10,.0f}")

---
## Report

### Factors Affecting Churn Risk

The final Log-Normal AFT model reveals that **customer category is the strongest predictor of tenure**. Subscribers in E-service, Total service, and Plus service packages survive 2.1–2.5× longer than Basic plan customers, suggesting that product depth drives loyalty. Sociodemographic factors also matter: older subscribers and those who have lived at the same address for longer tend to stay, reflecting life-stage stability that aligns with long-term contracts. On the other hand, internet add-on subscribers churn significantly faster — likely because internet-only or internet-heavy users are more price-sensitive and comparison-shop more actively. Unmarried status and the voice add-on also carry a modest accelerating effect on churn.

### Most Valuable Segments

A valuable customer, as defined here, is one who combines **high CLV** (high income, long survival probability) with **low churn risk within the next year**. By this definition, the most valuable segment is **Plus service and Total service subscribers who are older (45–65), married, and residentially stable**. Mean CLV for the Plus service segment is ~$64,000 versus ~$31,000 for Basic, and older (45–65) customers out-value younger ones by approximately 40%. The least valuable segment is young, unmarried Basic-plan subscribers with internet add-ons — they churn quickly and have lower margins.

### Retention Budget

With 27.4% overall churn and 11 subscribers at high risk of churning within 12 months (S(12m) < 0.5), the total CLV at stake is approximately **$115,000**. Using a 30% spend-to-save rule (a common heuristic: spend up to 30% of the value you'd recover), the suggested **annual retention budget is ≈ $34,500**. This budget should be concentrated on at-risk Plus/Total service customers, since their recovery value per head is highest.

### Additional Retention Suggestions

1. **Targeted internet-user campaigns** – since internet subscribers churn fastest, bundle offers (internet + voice + streaming discount) could reduce churn without cannibalising margin.
2. **Life-event triggers** – address changes are associated with lower loyalty; proactively reaching out during a move with a loyalty discount could prevent churn.
3. **Tier-upgrade nudges** – converting Basic subscribers to Plus/Total service is the single most impactful lever; model shows it extends lifetime by 2×.
4. **Age-segmented outreach** – younger (<30) subscribers have shorter tenures; early-tenure engagement programmes (onboarding, loyalty points) could shift the survival curve rightward for this group.
